# Gold Aggregation
Test the Gold layer aggregations independently.
Note: You should have run silver_transform first so there is pending Silver data.

In [ ]:
from datetime import datetime, timezone
import polars as pl
from electricity_maps.layers.gold import aggregate_gold_mix, aggregate_gold_flows
from electricity_maps.utils.state import PipelineState
from electricity_maps.config import get_settings

In [ ]:
settings = get_settings()
state = PipelineState(settings.data_dir)

now = datetime.now(timezone.utc)
gold_ts = int(now.timestamp())

In [ ]:
ts_list = state.pickup_ready("silver")
if not ts_list:
    print("No pending silver data found. Run 03_silver_transform first.")
else:
    print(f"Picked up Silver batches: {ts_list}")
    
    aggregate_gold_mix()
    aggregate_gold_flows()
    
    # Mark state
    state.mark_complete("silver", ts_list)
    state.init_layer("gold", gold_ts, now)
    state.mark_ready("gold", gold_ts, now, len(ts_list))
    
    print("Gold aggregation completed!")

In [ ]:
# Inspect Gold Mix Summary
print("Gold Daily Mix Summary:")
try:
    df_gold_mix = pl.read_delta(str(settings.gold_dir / "daily_mix_summary"))
    display(df_gold_mix.head())
except Exception as e:
    print("Error reading gold mix:", e)

In [ ]:
# Inspect Gold Daily Imports
print("\nGold Daily Imports:")
try:
    df_imports = pl.read_delta(str(settings.gold_dir / "daily_imports"))
    display(df_imports.head())
except Exception as e:
    print("Error reading gold imports:", e)